# Step 1: Data Preparation for Ad Generation

**Approach: product-first, fully automated selection**

Products and topics are selected by a deterministic scoring rule — no manual curation.

**Selection rules:**
- *Eligibility:* 2+ topics with 5+ positive AND 5+ negative reviews; non-empty metadata description; exclude non-guitar accessories
- *Product score:* `n_qualifying_topics + description_length / 1000` (richness first, description quality as tiebreaker)
- *Deduplication:* one product per brand (highest score retained)
- *Final selection:* top 3 products
- *Topic selection:* per product, pick 3 qualifying topics with smallest balance gap (|pct_positive − 50%|)

**Output:** `ad_generation_input.json`

## 1. Load Review Data

In [1]:
import pandas as pd
import json

df = pd.read_parquet('../drums_with_topics.parquet')
print(f'Total reviews: {len(df)}')
df.head(3)

Total reviews: 144598


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,subcategory,store,average_rating,price,topic_id,topic_label,topic_prob
66,5,Good choice,Very nice bells....a tad more than I wanted to...,[],B00MDNMVWW,B00MDNMVWW,AHQE5UGNEVWAJN6X7JN6RNQTIPWQ,2021-03-21 16:47:41.490,1,True,Drums & Percussion,AzureGreen,4.3,6.1,6.0,Small percussion accessories,0.813440
67,5,Omg,This is outstanding. Sounds perfect.,[],B08SL626JH,B0BCDVGKTT,AGTHQ6ANUEV7VOOVAEFWVMFIILUA,2021-10-24 01:56:21.777,0,True,Drums & Percussion,leize,4.5,79.99,NaN,None,NaN
68,5,Little but nice,Much smaller than I expected but it’s good,[],B07BZZWP8M,B0B1Z31ZL3,AGTHQ6ANUEV7VOOVAEFWVMFIILUA,2021-10-19 22:50:47.258,0,True,Drums & Percussion,HIMALAYAN BAZAAR,4.6,26.97,6.0,Small percussion accessories,0.687997


## 2. Load Product Metadata

Provides `product_title`, `description`, `features` for LLM prompts.  
Join key: `parent_asin`.

In [7]:
print('Loading metadata... (may take ~30 seconds)')

meta_lookup = {}
with open('../meta_Musical_Instruments.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        pa = obj.get('parent_asin')
        if pa:
            desc = obj.get('description', '')
            if isinstance(desc, list):
                desc = ' '.join(desc)
            meta_lookup[pa] = {
                'product_title': obj.get('title', ''),
                'description':   desc,
                'features':      obj.get('features', []),
                'store':         obj.get('store', ''),
                'price':         obj.get('price'),
            }

print(f'Metadata entries loaded: {len(meta_lookup)}')

Loading metadata... (may take ~30 seconds)
Metadata entries loaded: 213593


## 3. Derive Sentiment from Rating

- Rating 4–5 → **positive** (Strategy A input)
- Rating 1–2 → **negative** (Strategy B input)
- Rating 3 → dropped

In [8]:
df = df[df['rating'] != 3].copy()
df['sentiment'] = df['rating'].apply(lambda r: 'positive' if r >= 4 else 'negative')

print(f'Reviews after dropping rating=3: {len(df)}')
print(df['sentiment'].value_counts())

Reviews after dropping rating=3: 135203
sentiment
positive    119119
negative     16084
Name: count, dtype: int64


## 4. Find All Qualifying Product × Topic Combinations

A combination qualifies if it has **5+ positive AND 5+ negative** reviews.

In [9]:
MIN_PER_SENTIMENT = 5

combos = []
for (asin, topic), group in df.groupby(['asin', 'topic_label']):
    n_pos = (group['sentiment'] == 'positive').sum()
    n_neg = (group['sentiment'] == 'negative').sum()
    if n_pos >= MIN_PER_SENTIMENT and n_neg >= MIN_PER_SENTIMENT:
        combos.append({
            'asin':        asin,
            'topic':       topic,
            'n_pos':       int(n_pos),
            'n_neg':       int(n_neg),
            'total':       int(n_pos + n_neg),
            'pct_pos':     round(n_pos / (n_pos + n_neg) * 100, 1),
            'balance_gap': round(abs(n_pos / (n_pos + n_neg) - 0.5) * 100, 1),
            'parent_asin': group['parent_asin'].iloc[0],
            'store':       group['store'].iloc[0],
            'avg_rating':  group['average_rating'].iloc[0],
        })

combo_df = pd.DataFrame(combos)
print(f'Qualifying product x topic combinations: {len(combo_df)}')
print(f'Unique products: {combo_df["asin"].nunique()}')
print(f'\nTopics represented:')
print(combo_df['topic'].value_counts())

Qualifying product x topic combinations: 271
Unique products: 212

Topics represented:
topic
Product defects / returns          118
Electronic drums / practice pad     43
Small percussion accessories        35
Singing bowls / meditation          32
Cymbals                             17
Beginner / ease of play              9
Drum hardware / shells               9
Gift / holiday purchase              8
Name: count, dtype: int64


## 5. Automated Product Selection

Scoring rule → deduplication → top 3.

In [10]:
# --- Config ---
N_PRODUCTS          = 3    # number of products to select
N_TOPICS_PER_PROD   = 3    # number of topics per product
SAMPLE_SIZE         = 10   # max reviews to sample per topic per sentiment
EXCLUDE_ASINS       = ['B004GEW3H4']  # non-guitar accessories

# Build product-level summary
product_summary = combo_df.groupby('asin').agg(
    n_topics    = ('topic',  'count'),
    store       = ('store',  'first'),
    avg_rating  = ('avg_rating', 'first'),
    parent_asin = ('parent_asin', 'first'),
).reset_index()

product_summary['desc']     = product_summary['parent_asin'].apply(
    lambda pa: meta_lookup.get(pa, {}).get('description', ''))
product_summary['has_desc'] = product_summary['desc'].apply(
    lambda d: bool(d.strip()))
product_summary['desc_len'] = product_summary['desc'].apply(len)
product_summary['title']    = product_summary['parent_asin'].apply(
    lambda pa: meta_lookup.get(pa, {}).get('product_title', ''))

# Apply eligibility filter
eligible = product_summary[
    (product_summary['n_topics'] >= 2) &
    (product_summary['has_desc']) &
    (~product_summary['asin'].isin(EXCLUDE_ASINS))
].copy()

# Score and sort
eligible['score'] = eligible['n_topics'] + eligible['desc_len'] / 1000
eligible = eligible.sort_values('score', ascending=False).reset_index(drop=True)

# One-per-brand deduplication
selected_rows = []
used_brands   = set()
for _, row in eligible.iterrows():
    if row['store'] not in used_brands:
        selected_rows.append(row)
        used_brands.add(row['store'])
    if len(selected_rows) == N_PRODUCTS:
        break

selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)

print(f'=== Selected {N_PRODUCTS} products ===')
print(selected_df[['asin', 'store', 'avg_rating', 'n_topics', 'score',
                    'title']].to_string())

=== Selected 3 products ===
         asin                              store  avg_rating  n_topics  score                                                                                                                                                          title
0  B0000775G0                   Woodstock Chimes         4.8         5  7.381                  Woodstock Chimes Home of The Original Guaranteed Musically Tuned Wind Zenergy Hand Chime for Classrooms Meditation Mindfulness and More, Solo
1  B0187KO8X4                             Alesis         4.5         2  6.797                                                                                 Alesis Nitro Kit | Electronic Drum Set with 8" Snare, 8" Toms, and 10" Cymbals
2  B001O5VZ0E  TM THAMELMART FOR BEAUTIFUL MINDS         4.6         3  4.353  2.5” Tingsha Bell Cymbals Set - Om Nama Shivay Embossed Tibetan Chimes - Great for Yoga, Meditation, Spiritual, Mindfulness or Relaxation - Handmade in Nepal


## 6. Automated Topic Selection

For each selected product, pick the `N_TOPICS_PER_PROD` qualifying topics
with the smallest balance gap (pct_positive closest to 50%).

In [12]:
selected_topics = {}  # asin -> list of topic names

print('=== Selected topics per product ===\n')
for _, row in selected_df.iterrows():
    product_combos = combo_df[combo_df['asin'] == row['asin']].sort_values('balance_gap')
    chosen = product_combos.head(N_TOPICS_PER_PROD)
    selected_topics[row['asin']] = chosen['topic'].tolist()

    print(f"{row['asin']} | {row['store']} | {row['title'][:55]}")
    for _, c in chosen.iterrows():
        print(f"  {c['topic']:<35} pct_pos={c['pct_pos']:5.1f}%  "
              f"pos={c['n_pos']:3d}  neg={c['n_neg']:3d}  gap={c['balance_gap']:.1f}pp")
    print()

# Cross-product topic overlap
topic_sets = [set(v) for v in selected_topics.values()]
shared_all = topic_sets[0] & topic_sets[1] & topic_sets[2]
print(f'Topics shared across all 3 products: {shared_all if shared_all else "none"}')

total_ads = sum(len(v) for v in selected_topics.values()) * 2
print(f'\nTotal ads to generate: {total_ads}')

=== Selected topics per product ===

B0000775G0 | Woodstock Chimes | Woodstock Chimes Home of The Original Guaranteed Musica
  Product defects / returns           pct_pos= 45.0%  pos= 36  neg= 44  gap=5.0pp
  Cymbals                             pct_pos= 75.0%  pos= 15  neg=  5  gap=25.0pp
  Small percussion accessories        pct_pos= 90.5%  pos= 86  neg=  9  gap=40.5pp

B0187KO8X4 | Alesis | Alesis Nitro Kit | Electronic Drum Set with 8" Snare, 8
  Product defects / returns           pct_pos= 50.0%  pos= 11  neg= 11  gap=0.0pp
  Electronic drums / practice pad     pct_pos= 87.2%  pos=163  neg= 24  gap=37.2pp

B001O5VZ0E | TM THAMELMART FOR BEAUTIFUL MINDS | 2.5” Tingsha Bell Cymbals Set - Om Nama Shivay Embossed
  Product defects / returns           pct_pos= 31.8%  pos= 14  neg= 30  gap=18.2pp
  Small percussion accessories        pct_pos= 81.8%  pos= 27  neg=  6  gap=31.8pp
  Singing bowls / meditation          pct_pos= 93.6%  pos=102  neg=  7  gap=43.6pp

Topics shared across all 3 

## 7. Sample Reviews and Build Output JSON

For each selected product × topic:
- Up to 10 positive reviews → Strategy A input
- Up to 10 negative reviews → Strategy B input

Sentiment ratios are **product-specific** (not market-wide).

In [13]:
output = []

for _, row in selected_df.iterrows():
    asin      = row['asin']
    pa        = row['parent_asin']
    meta      = meta_lookup.get(pa, {})
    group     = df[df['asin'] == asin]
    topics    = selected_topics[asin]

    product_entry = {
        'asin':           asin,
        'product_title':  meta.get('product_title', ''),
        'brand':          row['store'],
        'average_rating': float(row['avg_rating']) if pd.notna(row['avg_rating']) else None,
        'description':    meta.get('description', ''),
        'features':       meta.get('features', []),
        'topics':         {}
    }

    for topic in topics:
        topic_reviews = group[group['topic_label'] == topic]
        pos_reviews   = topic_reviews[topic_reviews['sentiment'] == 'positive']['text'].dropna()
        neg_reviews   = topic_reviews[topic_reviews['sentiment'] == 'negative']['text'].dropna()

        n_pos = len(pos_reviews)
        n_neg = len(neg_reviews)
        pct_pos = round(n_pos / (n_pos + n_neg) * 100, 1)

        product_entry['topics'][topic] = {
            'pct_positive':     pct_pos,
            'pct_negative':     round(100 - pct_pos, 1),
            'n_positive_total': int(n_pos),
            'n_negative_total': int(n_neg),
            'positive_reviews': pos_reviews.sample(
                min(SAMPLE_SIZE, n_pos), random_state=42).tolist(),
            'negative_reviews': neg_reviews.sample(
                min(SAMPLE_SIZE, n_neg), random_state=42).tolist(),
        }

    output.append(product_entry)
    print(f"Processed: {asin} | {product_entry['product_title'][:60]}")

print(f'\nTotal products: {len(output)}')
print(f'Total ads to generate: {sum(len(p["topics"])*2 for p in output)}')

Processed: B0000775G0 | Woodstock Chimes Home of The Original Guaranteed Musically T
Processed: B0187KO8X4 | Alesis Nitro Kit | Electronic Drum Set with 8" Snare, 8" Tom
Processed: B001O5VZ0E | 2.5” Tingsha Bell Cymbals Set - Om Nama Shivay Embossed Tibe

Total products: 3
Total ads to generate: 16


## 8. Save Output

In [14]:
with open('ad_generation_input.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print('Saved to ad_generation_input.json\n')
print('--- Sanity Check ---')
for p in output:
    print(f"\n{p['asin']} | {p['product_title'][:55]} | ★{p['average_rating']}")
    print(f"  Desc: {p['description'][:120]}...")
    for topic, data in p['topics'].items():
        print(f"  [{topic}] "
              f"{data['n_positive_total']} pos / {data['n_negative_total']} neg total  "
              f"→ sampled {len(data['positive_reviews'])} / {len(data['negative_reviews'])}  "
              f"| pct_pos={data['pct_positive']}%")

Saved to ad_generation_input.json

--- Sanity Check ---

B0000775G0 | Woodstock Chimes Home of The Original Guaranteed Musica | ★4.8
  Desc: Product Description We live in a vast ocean of sound. But what is the sound made by a single drop of water? Just a gentl...
  [Product defects / returns] 36 pos / 44 neg total  → sampled 10 / 10  | pct_pos=45.0%
  [Cymbals] 15 pos / 5 neg total  → sampled 10 / 5  | pct_pos=75.0%
  [Small percussion accessories] 86 pos / 9 neg total  → sampled 10 / 9  | pct_pos=90.5%

B0187KO8X4 | Alesis Nitro Kit | Electronic Drum Set with 8" Snare, 8 | ★4.5
  Desc: The Alesis Nitro is a complete 8 piece electronic drum kit that includes everything you need to play like a pro. It feat...
  [Product defects / returns] 11 pos / 11 neg total  → sampled 10 / 10  | pct_pos=50.0%
  [Electronic drums / practice pad] 163 pos / 24 neg total  → sampled 10 / 10  | pct_pos=87.2%

B001O5VZ0E | 2.5” Tingsha Bell Cymbals Set - Om Nama Shivay Embossed | ★4.6
  Desc: ★ 2.5” Tingsh

## 9. Preview One Entry

Verify the JSON structure before moving to Step 2 (ad generation).

In [15]:
first       = output[0]
first_topic = list(first['topics'].keys())[0]
data        = first['topics'][first_topic]

print(f"Product : {first['product_title']}")
print(f"Brand   : {first['brand']}  |  Rating: {first['average_rating']}")
print(f"\nDescription:\n{first['description'][:400]}")
print(f"\nFeatures:")
for feat in first['features'][:3]:
    print(f'  • {feat}')

print(f"\n{'='*60}")
print(f"Topic: {first_topic}")
print(f"Product-level sentiment: {data['pct_positive']}% pos / {data['pct_negative']}% neg")
print(f"Total in topic: {data['n_positive_total']} pos, {data['n_negative_total']} neg")

print(f"\n--- Positive reviews (Strategy A input) ---")
for r in data['positive_reviews'][:2]:
    print(f'  • {r[:250]}')

print(f"\n--- Negative reviews (Strategy B input) ---")
for r in data['negative_reviews'][:2]:
    print(f'  • {r[:250]}')

Product : Woodstock Chimes Home of The Original Guaranteed Musically Tuned Wind Zenergy Hand Chime for Classrooms Meditation Mindfulness and More, Solo
Brand   : Woodstock Chimes  |  Rating: 4.8

Description:
Product Description We live in a vast ocean of sound. But what is the sound made by a single drop of water? Just a gentle tap with the mallet and the Zenergy Chime emits a powerful tone of singular beauty that lasts and lasts. Sounds like this are often used in meditation and healing, because they help us to focus and redirect our attention to the sounds within. The resonating sound not only calms

Features:
  • High Quality Beautiful Sound - Each Zenergy Chime note is tuned to clear tones and lovely harmonies. A gentle tap of the included mallet emits powerful tones of singular beauty that last and last - a great meditation chime, a great classroom chime
  • Great for Classrooms, Meditation and Healing - Sounds omitted by the Zenergy Chime - Solo, Silver are often used in meditat